In [1]:
!pip install "https://github.com/Dao-AILab/flash-attention/releases/download/v2.8.3/flash_attn-2.8.3%2Bcu12torch2.9cxx11abiTRUE-cp312-cp312-linux_x86_64.whl"

: 

In [2]:
!rm -rf "/content/SimpleVLLM"

In [3]:
!git clone https://github.com/Ajax0564/SimpleVLLM.git

Cloning into 'SimpleVLLM'...
remote: Enumerating objects: 122, done.
remote: Counting objects: 100% (122/122), done.
remote: Compressing objects: 100% (69/69), done.
remote: Total 122 (delta 53), reused 103 (delta 34), pack-reused 0 (from 0)
Receiving objects: 100% (122/122), 127.75 KiB | 25.55 MiB/s, done.
Resolving deltas: 100% (53/53), done.


In [4]:
import os
import sys

# Point directly to the 'src' directory
src_path = os.path.abspath("/content/SimpleVLLM/src")
if src_path not in sys.path:
    sys.path.append(src_path)

print("Successfully added to sys.path:", src_path)

Successfully added to sys.path: /content/SimpleVLLM/src


In [5]:
import sys

# Purge all simplevllm modules from Python's cache
for module in list(sys.modules.keys()):
    if module.startswith("simplevllm"):
        del sys.modules[module]

# Re-import core engine and model functions
from simplevllm.engine import ContinuousBatchEngine, PagedKVManager, SequenceState
from simplevllm.models import Qwen3Config, get_qwen3_model, get_qwen3_tokenizer

print("✓ Successfully imported Qwen3Config and SimpleVLLM modules!")
print("Config max_batch_size:", Qwen3Config["max_batch_size"])

✓ Successfully imported Qwen3Config and SimpleVLLM modules!
Config max_batch_size: 8


In [6]:
# Load model and tokenizer
tokenizer = get_qwen3_tokenizer()
model = get_qwen3_model()

print("✓ Model and Tokenizer initialized on device successfully!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


✓ Model and Tokenizer initialized on device successfully!


In [7]:
import torch
torch.manual_seed(123)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

Qwen3Model(
  (tok_emb): Embedding(151936, 1024)
  (trf_blocks): ModuleList(
    (0-27): 28 x TransformerBlock(
      (att): GroupedQueryAttention(
        (W_query): Linear(in_features=1024, out_features=2048, bias=False)
        (W_key): Linear(in_features=1024, out_features=1024, bias=False)
        (W_value): Linear(in_features=1024, out_features=1024, bias=False)
        (out_proj): Linear(in_features=2048, out_features=1024, bias=False)
        (q_norm): RMSNorm()
        (k_norm): RMSNorm()
      )
      (ff): FeedForward(
        (fc1): Linear(in_features=1024, out_features=3072, bias=False)
        (fc2): Linear(in_features=1024, out_features=3072, bias=False)
        (fc3): Linear(in_features=3072, out_features=1024, bias=False)
      )
      (norm1): RMSNorm()
      (norm2): RMSNorm()
    )
  )
  (final_norm): RMSNorm()
  (out_head): Linear(in_features=1024, out_features=151936, bias=False)
)

In [8]:
from simplevllm.models import Qwen3Config
mgr = PagedKVManager(Qwen3Config, max_blocks=32, device=device)

In [9]:
prompt = "Give me a short introduction to large language models."

input_token_ids1 = tokenizer.encode(prompt)

In [10]:
prompt = "Give me a short introduction to AI"

input_token_ids3 = tokenizer.encode(prompt)

In [11]:
prompt = "Give me a short introduction to Embeddings in nlp"

input_token_ids2 = tokenizer.encode(prompt)

In [12]:
batched_engine = ContinuousBatchEngine(model, mgr, Qwen3Config)
print("BatchedEngine ready")

BatchedEngine ready


In [13]:
my_prompts = [
    input_token_ids1,
    input_token_ids2,
    input_token_ids3
]
batched_engine.reset()
for prompt in my_prompts:
    batched_engine.add_sequence(prompt,max_gen_len = 512)


while batched_engine.waiting_room or batched_engine.active:
    finished = batched_engine.step()
     # Print results as they come in
    for sid, tokens in finished.items():
        print(f"Slot {sid} finished. Total length: {len(tokens)}")
        print(tokenizer.decode(tokens))
        print("-" * 30)

Slot 2 finished. Total length: 180
<|im_start|>user
Give me a short introduction to AI<|im_end|>
<|im_start|>assistant
<think>
Okay, the user wants a short introduction to AI. Let me start by defining what AI is. I should mention it's a technology that helps machines think and learn. Maybe include some key points like machine learning, natural language processing, and applications. Need to keep it concise but informative. Also, make sure to highlight the benefits and the importance of AI in modern life. Avoid technical jargon to keep it accessible. Let me check if I'm covering all important aspects without being too long. Alright, that should work.
</think>

AI, or artificial intelligence, is a technology that enables machines to think and learn from data. It uses algorithms and machine learning to analyze information and make decisions automatically. AI has applications in various fields, from healthcare to finance, and is transforming how we interact with the world.<|im_end|>
-------

In [14]:
! pip install gradio

In [15]:
import os
import threading
import uuid

import gradio as gr
import torch


class GradioRuntime:
    def __init__(self):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.tokenizer = tokenizer
        self.model = model

        max_blocks = int(os.getenv("SIMPLEVLLM_MAX_BLOCKS", "256"))
        self.kv_manager = PagedKVManager(
            Qwen3Config,
            max_blocks=max_blocks,
            device=self.device,
        )
        self.engine = ContinuousBatchEngine(
            self.model,
            self.kv_manager,
            Qwen3Config,
        )
        self.conversations: dict[str, list[dict[str, str]]] = {}
        self.lock = threading.Lock()

    def _encode_history(self, history: list[dict[str, str]]) -> list[int]:
        prompt_ids: list[int] = []
        for item in history:
            turn = f"<|im_start|>{item['role']}\n{item['content']}<|im_end|>\n"
            prompt_ids.extend(self.tokenizer.encode(turn, chat_wrapped=False))

        prompt_ids.extend(
            self.tokenizer.encode("<|im_start|>assistant\n", chat_wrapped=False)
        )
        return prompt_ids

    def stream(self, conversation_id: str | None, message: str, max_new_tokens: int):
        conversation_id = conversation_id or uuid.uuid4().hex
        history = list(self.conversations.get(conversation_id, []))
        history.append({"role": "user", "content": message})
        prompt_ids = self._encode_history(history)

        if len(prompt_ids) + max_new_tokens > Qwen3Config["context_length"]:
            raise ValueError("Conversation is too long for the model context window")

        with self.lock:
            self.engine.reset()
            self.engine.add_sequence(prompt_ids, max_gen_len=max_new_tokens)
            generated_ids: list[int] | None = None
            answer = ""

            while self.engine.waiting_room or self.engine.active:
                for event in self.engine.step_single():
                    answer += self.tokenizer.decode([event["token_id"]])
                    answer = answer.split("<|im_end|>", 1)[0]
                    yield conversation_id, answer
                    if event["finished"]:
                        generated_ids = event["tokens"][len(prompt_ids):]

        if generated_ids is None:
            raise RuntimeError("The engine did not produce a response")

        final_answer = self.tokenizer.decode(generated_ids)
        final_answer = final_answer.split("<|im_end|>", 1)[0].strip()
        if not final_answer:
            final_answer = answer.strip()

        history.append({"role": "assistant", "content": final_answer})
        self.conversations[conversation_id] = history
        if final_answer != answer:
            yield conversation_id, final_answer


def create_demo(runtime: GradioRuntime | None = None) -> gr.Blocks:
    runtime = runtime or GradioRuntime()

    def respond(message, history, conversation_id, max_new_tokens):
        if not message or not message.strip():
            yield "", history, conversation_id
            return

        display_history = list(history or [])
        display_history.extend(
            [
                {"role": "user", "content": message},
                {"role": "assistant", "content": ""},
            ]
        )
        try:
            for conversation_id, answer in runtime.stream(
                conversation_id, message.strip(), int(max_new_tokens)
            ):
                display_history = [
                    *display_history[:-1],
                    {**display_history[-1], "content": answer},
                ]
                yield "", display_history, conversation_id
        except Exception as error:
            display_history = [
                *display_history[:-1],
                {**display_history[-1], "content": f"Error: {error}"},
            ]
            yield "", display_history, conversation_id

    def clear_chat():
        return [], None

    with gr.Blocks(title="SimpleVLLM Chat") as demo:
        gr.Markdown("# SimpleVLLM Chat")
        chatbot = gr.Chatbot(type="messages", height=600)
        conversation_id = gr.State(value=None)
        with gr.Row():
            message = gr.Textbox(
                placeholder="Message SimpleVLLM...",
                show_label=False,
                scale=8,
            )
            send = gr.Button("Send", variant="primary", scale=1)
        with gr.Row():
            max_new_tokens = gr.Slider(
                minimum=1,
                maximum=1024,
                value=256,
                step=1,
                label="Max new tokens",
            )
            clear = gr.Button("New chat")

        inputs = [message, chatbot, conversation_id, max_new_tokens]
        outputs = [message, chatbot, conversation_id]
        send.click(respond, inputs=inputs, outputs=outputs)
        message.submit(respond, inputs=inputs, outputs=outputs)
        clear.click(clear_chat, outputs=[chatbot, conversation_id])

    return demo


def main() -> None:
    create_demo().queue().launch()


if __name__ == "__main__":
    main()

/tmp/ipython-input-1194468199.py:113: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(type="messages", height=600)


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1c9fac0a5c3ecba2c6.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
